# 22 - ניתוב מחדש וחלוקה מחדש של העומס

כל מחברת חוסן קודמת בפרויקט זה עונה על השאלה *האם הרשת מתפרקת?* - היא מסירה תחנה ומודדת פיצול (articulation points, נתח הרכיב הקשיר הגדול ביותר, ירידה ביעילות). זוהי תשובה טופולוגית. אין זו החוויה של הנוסע. נוסע שתחנתו נסגרה אינו הופך בדרך כלל לבלתי נגיש; הוא פשוט בוחר **מסלול ארוך יותר**. עלות הסגירה נמדדת אפוא ב**דקות נסיעה נוספות**, ורק במקרים הגרועים ביותר ב**אובדן קשירות**.

מחברת זו בונה את המודל החסר הזה. עבור כל אחת מ-N התחנות הקריטיות מבחינה מבנית היא:

1. מאתרת מדגם של זוגות מוצא-יעד (OD) שהמסלול המהיר ביותר שלהם עובר כיום **דרך** אותה תחנה;
2. מוחקת את התחנה מגרף זמני הנסיעה;
3. מחשבת מחדש את המסלול המהיר ביותר עבור כל אחד מהזוגות הללו ומודדת את **העיקוף בשניות**, וכן כמה זוגות הופכים ל**בלתי נגישים**;
4. מתעדת **אילו תחנות אחרות מופיעות כעת במסלולים החדשים** - התחנות שסופגות את הנסיעות שהוסטו.

התוצאה היא דירוג של תחנות לפי **עלות זמן-נוסע** ולא לפי טופולוגיה, והשוואה ישירה בין שני הדירוגים. הם אינם מסכימים זה עם זה, ואי-ההסכמה עצמה היא הממצא.

**שאלת המחקר:** *כאשר תחנה קריטית מושבתת, כמה זמן נסיעה נוסף הרשת אכן כופה, על מי, ואילו תחנות שכנות סופגות את העומס?*

זה נותן מענה לסעיף בשקופית כיווני ההמשך "מודל ריאלי של ניתוב מחדש סביב תחנה מושבתת".

## קלט

* `outputs/nb/18_travel_time_network/tables/edges_traveltime.csv` - שורה אחת לכל מקטע מכוון עם `from_stop, to_stop, median_travel_seconds, p25_travel_seconds, p75_travel_seconds, trip_frequency, n_observations`. **זהו הקלט ההכרחי** והוא מיוצר על ידי מחברת 18. בלעדיו אין כל מושג של "דקות", אלא רק של קפיצות (hops), וכל תכליתה של מחברת זו מתאיינת.
* `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - משמש אך ורק לבחירת התחנות שיושבתו (`approx_betweenness` הגבוה ביותר) ולאספקת שמות, קואורדינטות ואזורים.

לא נקרא כאן קובץ GTFS גולמי: הקובץ `stop_times.txt` בנפח 816 MB כבר עוכל לכדי זמני נסיעה במחברת 18.

## מחברות שחייבות לרוץ קודם

`01` -> `02` -> `04` (עבור דירוג הקריטיות) ו-**`18`** (עבור גרף זמני הנסיעה). דבר מלבד זאת.

## פלט (הכול תחת `outputs/nb/22_rerouting_model/`)

* `tables/rerouting_results.csv` - שורה אחת לכל תחנה מושבתת: `removed_stop, stop_name, mean_detour_seconds, median_detour_seconds, pairs_disconnected, reachable_share, top_absorbing_stops` (בתוספת עמודות אבחון).
* `tables/absorbing_stations.csv` - טבלה בפורמט ארוך: עבור כל תחנה שהוסרה, כל תחנה שקיבלה תנועה נוספת וכמה זוגות OD מנותבים מחדש עוברים כעת דרכה.
* `tables/rank_comparison.csv` - דירוג טופולוגי מול דירוג זמן-נוסע, לכל תחנה.
* `tables/detour_samples.csv` - העיקופים ברמת זוג OD בודד שמאחורי הצברים, כדי שניתן יהיה לבדוק מחדש את ההתפלגויות.
* `rerouting_summary.json` - מספרי הכותרת, כל הקבועים שנעשה בהם שימוש, וסטטיסטיקות ההתאמה בין הדירוגים.
* `figures/detour_cost_per_station.png`, `figures/topological_vs_passenger_time_rank.png`, `figures/detour_distribution.png`, `figures/absorbing_stations_worst.png`.

התיקיות המוקפאות והמצוטטות בדוח `outputs/tables`, `outputs/figures` ו-`outputs/rail` אינן נוגעות כלל.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את המחברת הן על עותק מקומי והן על Google Colab. הוא מגדיר את `_ensure(...)`, המתקין באמצעות pip רק את החבילות שחסרות בפועל (כך שהרצה חוזרת של המחברת זולה), ואת `find_repo_root()`, המטפס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר תיקיית ה-GTFS, ואם לא מצא - משכפל את המאגר אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את תיקיית הפלט הראשית של המחברת. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות שלבים וקבועי העלות

אנו מייבאים את חבילות החישוב המדעי ובנוסף את `scipy.sparse.csgraph`. **מדוע scipy ולא `networkx` עבור המסלולים הקצרים ביותר?** מחברת זו מריצה אלפי שאילתות Dijkstra על גרף בן כ-30,000 צמתים וכ-52,000 קשתות. `networkx` כתובה כולה ב-Python ודורשת כ-0.3 שניות לכל הרצת Dijkstra ממקור יחיד; `scipy.sparse.csgraph.dijkstra` מהודרת ועונה על שאילתת *ריבוי מקורות* (מאות מקורות בבת אחת) בכשנייה אחת. הפרש זה הוא שמאפשר את הניתוח הזה בכלל על מחשב נייד. הגרף עדיין נבנה ומדווח באמצעות טבלאות תואמות `networkx`, ולכן דבר במודל אינו משתנה.

כל הפרמטרים היקרים מרוכזים בתא יחיד זה:

| קבוע | ברירת מחדל | מה הוא עולה |
| --- | --- | --- |
| `TOP_N_STATIONS` | 30 | הרצת Dijkstra אחת מריבוי מקורות על כל הגרף **לכל תחנה**. ראו את ההערה להלן מדוע סריקה מלאה אינה בת-השגה. |
| `N_SOURCES` | 250 | חישוב הבסיס של כל-הזוגות-ממדגם באמצעות Dijkstra. הזיכרון הוא `N_SOURCES x n_nodes` עבור המרחקים (float64, כ-61 MB) ועוד אותה כמות עבור מטריצת הקודמים (predecessors) מסוג int32 (כ-30 MB). העלאה ל-1000 הייתה דורשת כ-360 MB. |
| `TARGETS_PER_SOURCE` | 80 | 250 x 80 = 20,000 זוגות OD מועמדים. אינו עולה דבר בזמן Dijkstra - המרחקים כבר חושבו - אלא רק בשחזור המסלולים. |
| `MAX_PAIRS_PER_STATION` | 400 | מגביל את עבודת הניתוב מחדש עבור תחנה מרכזית מאוד. כל זוג שמנותב מחדש דורש מעבר אחד על מערך הקודמים. |
| `DISCONNECT_PENALTY_S` | 3600 | מספר השניות הנזקפות לחובת זוג OD שהופך ל**בלתי נגיש**. זוהי הכרעה שיפוטית ולא מדידה - ראו את פרק המגבלות. |

**מדוע לא לסרוק את כל 30,463 התחנות?** כל תחנה שמוסרת מחייבת בנייה מחדש של המטריצה הדלילה והרצה חדשה של Dijkstra מריבוי מקורות (כ-1-2 שניות כאן), *וגם* שאינדקס תנועת המעבר ייבנה עבור כל צומת ולא רק עבור 30. סריקה מלאה היא אפוא בסדר גודל של 10-17 שעות חישוב, בתוספת מדגם OD גדול בהרבה שיעניק לכל תחנה מספיק תנועת מעבר כדי שתהיה ניתנת למדידה. הצמצום ל-top-N הוא החלטה תקציבית, והוא גם החלטה שניתן להגן עליה: תחנות שנושאות כמעט שום תנועת מעבר הן בעלות עלות עיקוף אפסית כמעט מעצם הבנייה, ולכן המסה המעניינת מצויה בראש ההתפלגות. עם זאת, משמעות הדבר היא שמחברת זו **אינה יכולה** לגלות תחנה בעלת betweenness נמוך שעלות זמן-הנוסע שלה גבוהה באופן מפתיע - זהו כתם עיוור ידוע, המצוין שוב בפרק המגבלות.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'scipy', 'matplotlib', 'seaborn')

import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '22_rerouting_model'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants: the entire cost of this notebook lives here -------
TOP_N_STATIONS        = 30      # stations shut down, one at a time
N_SOURCES             = 250     # origins in the OD sample
TARGETS_PER_SOURCE    = 80      # destinations drawn per origin -> 20,000 OD pairs
MAX_PAIRS_PER_STATION = 400     # cap on rerouted pairs evaluated per removed station
MIN_EDGE_SECONDS      = 1.0     # floor on an edge weight (see note in section 5)
DISCONNECT_PENALTY_S  = 3600.0  # seconds charged to an OD pair that becomes unreachable
TOP_ABSORBING         = 5       # absorbing stations named per removed station
RANDOM_SEED           = 42
FIG_DPI               = 150

rng = np.random.default_rng(RANDOM_SEED)

print('stage folder :', STAGE)
print('OD sample    :', N_SOURCES, 'origins x', TARGETS_PER_SOURCE, 'destinations =',
      N_SOURCES * TARGETS_PER_SOURCE, 'candidate pairs')

## 3. רינדור תוויות בעברית

שמות התחנות בפיד ה-GTFS הישראלי הם בעברית, וכמה מהאיורים שלהלן מדפיסים אותם (תרשים העיקוף לכל תחנה, פיזור השוואת הדירוגים, ותרשים התחנות הסופגות). Matplotlib אינה מממשת את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני שהיא מצוירת, ובוחר גופן שאכן מכיל גליפים עבריים. התא אידמפוטנטי - הרצה חוזרת שלו לא תערים patches זה על זה (patch כפול היה הופך את הטקסט בחזרה).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור השלבים הקודמים

תיקיות השלבים מאותרות לפי **הקידומת הדו-ספרתית** שלהן (`OUT.glob('18*')`) ולא לפי slug מדויק, כך ששינוי שם קטן במחברת קודמת אינו שובר את זו. אם ארטיפקט נדרש חסר, פונקציית העזר מעלה `FileNotFoundError` המציין את שם המחברת שיש להריץ, במקום להיכשל בשלב מאוחר יותר עם `KeyError` סתום על עמודה חסרה.

In [ ]:
# --- Resolve previous stages by two-digit prefix ---------------------------
def stage_dir(prefix, notebook_hint):
    '''Return the output folder of a previous stage, matched by numeric prefix.'''
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            'No stage folder starting with ' + prefix + ' under ' + str(OUT) +
            ' - run notebook ' + notebook_hint + ' first.')
    return matches[0]


def stage_file(prefix, notebook_hint, filename):
    '''Return the path of `filename` inside a previous stage folder.'''
    folder = stage_dir(prefix, notebook_hint)
    direct = folder / 'tables' / filename
    if direct.exists():
        return direct
    hits = sorted(folder.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            filename + ' not found under ' + str(folder) +
            ' - run notebook ' + notebook_hint + ' first; it writes ' + filename + '.')
    return hits[0]


print('18 ->', stage_dir('18', '18_travel_time_network'))
print('04 ->', stage_dir('04', '04_centrality_analysis'))

## 5. גרף זמני הנסיעה ממחברת 18

אנו קוראים את `edges_traveltime.csv`, שעמודת `median_travel_seconds` שבו היא זמן הנסיעה החציוני שנצפה בין שתי תחנות עוקבות על פני כל הנסיעות שמשרתות את אותו מקטע. השימוש ב**חציון** ולא בממוצע הוא החלטה של מחברת 18, והיא ההחלטה הנכונה: לוחות הזמנים של GTFS מכילים לעתים פערים פתולוגיים (המתנות שקודדו כזמן נסיעה) שהיו מושכים ממוצע כלפי מעלה.

שלושה שינויים מכוונים מתרחשים כאן.

1. **הטלה לגרף לא-מכוון.** תחנה מושבתת חוסמת תנועה בשני הכיוונים, ושאלת החוסן היא סימטרית, ולכן אנו מכווצים כל זוג מכוון `(u,v)` / `(v,u)` לכדי קשת לא-מכוונת אחת. המשקל שנשמר הוא ה**מינימום** מבין שני הכיוונים - הדרך המהירה ביותר לחצות פיזית את המקטע. לקיחת הממוצע הייתה ממציאה זמן נסיעה שאף שירות אינו מציע בפועל.
2. **צבירת כפילויות לפני בניית המטריצה הדלילה.** `scipy.sparse.csr_matrix` **מסכמת** רשומות `(i,j)` כפולות. אילו היינו מוסרים לה שתי שורות עבור אותו זוג, משקל הקשת היה מוכפל בשקט. צבירה מוקדמת באמצעות `groupby` מונעת זאת לחלוטין.
3. **רצפה של שנייה אחת (`MIN_EDGE_SECONDS`).** במטריצה דלילה, אפס מאוחסן *הוא* רשומה חסרה: קשת במשקל `0` הייתה נחשבת בעיני `csgraph` כ**היעדר קשת כלל**, ובכך מנתקת בשקט תחנות החולקות את אותה דקה בלוח הזמנים (הדבר אכן קורה ב-GTFS כאשר שתי תחנות רשומות באותו `arrival_time`). קיבוע לשנייה אחת שומר עליהן מחוברות בעלות זניחה ביחס לזמני נסיעה הנמדדים בעשרות דקות. מספר הקשתות שקובעו מודפס כדי שסדר הגודל של ההתאמה יהיה גלוי.

In [ ]:
# --- Load the travel-time edge list written by notebook 18 -----------------
tt_path = stage_file('18', '18_travel_time_network', 'edges_traveltime.csv')
tt = pd.read_csv(tt_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')

needed = {'from_stop', 'to_stop', 'median_travel_seconds'}
absent = needed - set(tt.columns)
if absent:
    raise KeyError(
        str(tt_path) + ' is missing column(s) ' + str(sorted(absent)) +
        ' - expected the notebook 18 contract '
        '[from_stop, to_stop, median_travel_seconds, ...]. Found: ' + str(sorted(tt.columns)))

before = len(tt)
tt = tt.dropna(subset=['median_travel_seconds']).copy()
tt = tt[tt['from_stop'].astype(str) != tt['to_stop'].astype(str)]
raw_seconds = tt['median_travel_seconds'].astype(float)
n_clamped = int((raw_seconds < MIN_EDGE_SECONDS).sum())
tt['seconds'] = raw_seconds.clip(lower=MIN_EDGE_SECONDS)

# Undirected projection: one row per unordered pair, weight = fastest direction.
pair = np.sort(tt[['from_stop', 'to_stop']].to_numpy().astype(str), axis=1)
tt['a'] = pair[:, 0]
tt['b'] = pair[:, 1]
und = (tt.groupby(['a', 'b'], as_index=False)
         .agg(seconds=('seconds', 'min'), n_directions=('seconds', 'size')))

print('travel-time edges read  :', format(before, ','), '<-', tt_path)
print('usable directed edges   :', format(len(tt), ','),
      '(' + format(before - len(tt), ',') + ' dropped: missing time or self-loop)')
print('undirected edges        :', format(len(und), ','))
print('edges clamped to', MIN_EDGE_SECONDS, 'second :', format(n_clamped, ','),
      '(' + format(100 * n_clamped / max(len(tt), 1), '.2f') + '% of usable directed rows)')
print()
print(und['seconds'].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))

## 6. בניית הגרף הדליל ואיתור הרכיב הענק

מזהי התחנות ממופים לאינדקסים שלמים `0..n-1`, ורשימת הקשתות הלא-מכוונת נכתבת אל תוך מטריצת CSR סימטרית - הייצוג ש-`scipy.sparse.csgraph` דורשת. לאחר מכן אנו מתייגים את הרכיבים הקשירים ושומרים את **הרכיב הענק** כמרחב הדגימה של זוגות OD.

הדגימה בתוך הרכיב הענק בלבד אינה עניין קוסמטי: זוג OD הנדגם בין שני רכיבים שונים הוא בלתי נגיש *עוד לפני* שהוסרה תחנה כלשהי, ולכן היה תורם בסיס אינסופי חסר משמעות ומזהם הן את סטטיסטיקות העיקוף והן את ספירת "הזוגות שנותקו". ההגבלה על המדגם מבטיחה שכל ניתוק שמדווח **נגרם** באמת על ידי ההסרה.

In [ ]:
# --- Integer indexing and the symmetric CSR matrix -------------------------
node_ids = np.unique(np.concatenate([und['a'].to_numpy(), und['b'].to_numpy()]))
index_of = {sid: i for i, sid in enumerate(node_ids)}
n_nodes = len(node_ids)

ai = und['a'].map(index_of).to_numpy(dtype=np.int32)
bi = und['b'].map(index_of).to_numpy(dtype=np.int32)
wt = und['seconds'].to_numpy(dtype=float)

rows = np.concatenate([ai, bi])
cols = np.concatenate([bi, ai])
data = np.concatenate([wt, wt])

BASE = csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))

n_comp, labels = connected_components(BASE, directed=False)
sizes = np.bincount(labels)
giant_label = int(sizes.argmax())
giant_nodes = np.flatnonzero(labels == giant_label)

print('nodes in travel-time graph :', format(n_nodes, ','))
print('undirected edges           :', format(len(und), ','))
print('connected components       :', n_comp)
print('giant component            :', format(len(giant_nodes), ','), 'nodes',
      '(' + format(100 * len(giant_nodes) / n_nodes, '.2f') + '% of the graph)')

## 7. בחירת התחנות שיושבתו

רשימת המועמדות היא **`TOP_N_STATIONS` התחנות המובילות לפי `approx_betweenness`** ממחברת 04, מוגבלת לתחנות הקיימות ברכיב הענק של גרף זמני הנסיעה. Betweenness היא ההגדרה הטופולוגית הטבעית של "קריטיות": היא סופרת כמה מסלולים קצרים ביותר עוברים דרך תחנה. השימוש בה כאן מכוון - כל תכליתה של המחברת היא לקחת את הרשימה הטופולוגית המצומצמת ולשאול אם היא שורדת מפגש עם מדידת זמן-נוסע.

שימו לב לשם העמודה: מחברת 04 מייצאת `approx_betweenness` (קירוב המחושב ממדגם pivots), ולא `betweenness`. הטוען מקבל כל אחת משתי האיותים אך אינו ממציא אחת מהן.

תחנות ממחברת 04 שחסרות בגרף של מחברת 18 מדווחות ולא מושמטות בשקט: אלה תחנות המופיעות בגרף שכנויות הנסיעות אך לא ניתן היה למדוד עבורן זמן נסיעה תקף, והיעדרן מהווה פער כיסוי אמיתי (וקטן).

In [ ]:
# --- Candidate stations: top-N by approximate betweenness ------------------
metrics_path = stage_file('04', '04_centrality_analysis', 'stop_metrics.csv')
metrics = pd.read_csv(metrics_path, dtype={'stop_id': str}, encoding='utf-8-sig')

btw_col = next((c for c in ('approx_betweenness', 'betweenness') if c in metrics.columns), None)
if btw_col is None:
    raise KeyError(
        str(metrics_path) + ' has no approx_betweenness column - run notebook 04 '
        '(centrality analysis) first. Found: ' + str(sorted(metrics.columns)))
for col in ('stop_name', 'region'):
    if col not in metrics.columns:
        metrics[col] = ''

giant_ids = set(node_ids[giant_nodes].tolist())
ranked = metrics.sort_values(btw_col, ascending=False).reset_index(drop=True)
ranked['topological_rank'] = np.arange(1, len(ranked) + 1)

in_graph = ranked[ranked['stop_id'].isin(giant_ids)]
candidates = in_graph.head(TOP_N_STATIONS).copy().reset_index(drop=True)
# Rank within the shortlist, so the comparison later is 1..N against 1..N.
candidates['topological_rank'] = np.arange(1, len(candidates) + 1)

dropped = ranked.head(TOP_N_STATIONS)[~ranked.head(TOP_N_STATIONS)['stop_id'].isin(giant_ids)]
print('candidate stations         :', len(candidates))
print('top-N stops absent from the travel-time giant component:', len(dropped))
if len(dropped):
    print('  ', list(dropped['stop_id'])[:10])

name_of = dict(zip(metrics['stop_id'].astype(str), metrics['stop_name'].fillna('').astype(str)))
candidates[['stop_id', 'stop_name', 'region', btw_col, 'topological_rank']].head(10)

## 8. הבסיס: דגימת זוגות OD ואיתור אלה המשתמשים בכל מועמדת

זהו לב המודל והתא היקר ביותר מבחינה חישובית.

1. `N_SOURCES` מקורות נדגמים באופן אחיד ואקראי מתוך הרכיב הענק, ו**Dijkstra יחיד מריבוי מקורות** מחשב, בקוד מהודר, את זמן הנסיעה המהיר ביותר מכל מוצא לכל תחנה, יחד עם מטריצת קודמים (predecessor matrix).
2. עבור כל מוצא נדגמים `TARGETS_PER_SOURCE` יעדים מתוך התחנות שאליהן הוא יכול להגיע.
3. המסלול המהיר ביותר של כל זוג משוחזר על ידי מעבר לאחור על מערך הקודמים החל מהיעד.
4. אנו שומרים **רק את הזוגות שמסלולם עובר דרך לפחות תחנה מועמדת אחת כתחנת ביניים**. אלה הנסיעות שסגירה אכן הייתה משבשת; כל השאר אינו רלוונטי לניסוי זה ונזרק מיד, וזה מה ששומר על צריכת זיכרון נמוכה (אנו מאחסנים מסלול `int32` אחד לכל זוג שנשמר, ולא לכל זוג שנדגם).

שיעור הזוגות הנדגמים ששורדים את שלב 4 הוא כשלעצמו מספר כותרת: הוא מציין איזה חלק מהנסיעות האקראיות בישראל תלוי באחת מ-30 התחנות המרכזיות ביותר.

הסתייגות שיש להצהיר עליה מראש: **זוגות OD נדגמים באופן אחיד על פני תחנות, ולא על פני נוסעים.** זוג בין שתי תחנות כפריות נספר בדיוק כמו זוג בין שני מסופי החלפה בתל אביב. הדבר הופך את התוצאות למדד *מבני-רשתי* של זמן נוסע, ולא למדד משוקלל ביקוש. מחברת 21 בונה את הפרוקסי לביקוש המבוסס אוכלוסייה שהיה נדרש כדי לשקלל אותם; שילוב השניים הוא הצעד הבא המתבקש ואינו נעשה כאן.

In [ ]:
# --- Baseline multi-source Dijkstra and the through-traffic index ----------
def reconstruct(pred_row, source_i, target_i):
    '''Walk a scipy predecessor row backwards; return the node path or None.'''
    if target_i == source_i:
        return [int(source_i)]
    path = [int(target_i)]
    cur = int(target_i)
    while cur != source_i:
        cur = int(pred_row[cur])
        if cur < 0:                      # scipy marks no-predecessor as -9999
            return None
        path.append(cur)
    path.reverse()
    return path


cand_index = np.array([index_of[s] for s in candidates['stop_id']], dtype=np.int64)
cand_set = set(int(i) for i in cand_index)

n_src = min(N_SOURCES, len(giant_nodes))
sources = rng.choice(giant_nodes, size=n_src, replace=False)

t0 = time.time()
dist0, pred0 = dijkstra(BASE, directed=False, indices=sources, return_predecessors=True)
print('baseline Dijkstra:', format(time.time() - t0, '.1f'), 's for', n_src, 'origins')

pairs = []        # (source_idx, target_idx, baseline_seconds) for retained pairs only
old_paths = []    # int32 path array, aligned with `pairs`
through = {int(c): [] for c in cand_index}
n_sampled = 0

for r, s_i in enumerate(sources):
    s_i = int(s_i)
    reach = np.flatnonzero(np.isfinite(dist0[r]))
    reach = reach[reach != s_i]
    if reach.size == 0:
        continue
    k = min(TARGETS_PER_SOURCE, reach.size)
    targets = rng.choice(reach, size=k, replace=False)
    prow = pred0[r]
    for t_i in targets:
        t_i = int(t_i)
        n_sampled += 1
        path = reconstruct(prow, s_i, t_i)
        if path is None or len(path) < 3:
            continue                      # no interior stop: nothing to route through
        hits = {nd for nd in path[1:-1] if nd in cand_set}
        if not hits:
            continue
        p = len(pairs)
        pairs.append((s_i, t_i, float(dist0[r, t_i])))
        old_paths.append(np.asarray(path, dtype=np.int32))
        for nd in hits:
            through[nd].append(p)

path_lengths = np.array([len(p) for p in old_paths]) if old_paths else np.array([0])
print('OD pairs sampled                    :', format(n_sampled, ','))
print('pairs routed through a candidate    :', format(len(pairs), ','),
      '(' + format(100 * len(pairs) / max(n_sampled, 1), '.1f') + '% of the sample)')
print('median stops on a retained route    :', int(np.median(path_lengths)))
print('candidates with zero through-traffic:',
      sum(1 for v in through.values() if not v), 'of', len(through))

## 9. ניסוי הניתוב מחדש

`reroute_one` מבצעת את ההשבתה בפועל עבור תחנה בודדת:

* **הסרת התחנה.** כל רשומה בשלשת ה-COO הנוגעת לאינדקס הצומת הזה ממוסכת החוצה, ומטריצת CSR חדשה נבנית. הצומת נשאר במטריצה אך מבודד, כך שכל האינדקסים נותרים תקפים - ללא תיוג מחדש וללא באגי ניהול רשומות.
* **ניתוב מחדש.** זוגות ה-OD המושפעים מקובצים לפי מוצא, ו-Dijkstra יחיד מריבוי מקורות מורץ על הגרף הפגוע עבור בדיוק אותם מוצאים הזקוקים לכך.
* **מדידת העיקוף** כ-`new_seconds - baseline_seconds`. ערך זה אי-שלילי מעצם הבנייה (מחיקת קשתות לעולם אינה יכולה להאיץ מסלול); הוא נקטם באפס בכל מקרה כדי לספוג רעש נקודה צפה.
* **ספירת הזוגות הבלתי נגישים** - אלה שמרחקם החדש אינסופי. אלה הניתוקים האמיתיים, והם מוצאים מממוצע העיקוף משום שלעיקוף אינסופי אין ממוצע. הם מדווחים בנפרד כ-`pairs_disconnected` / `reachable_share`, ומשולבים בעלות המשוקללת בהמשך באמצעות קנס מפורש.
* **ייחוס העומס שהוסט.** כל תחנה המופיעה במסלול ה*חדש* אך **לא** במסלול ה*ישן* נזקפת לזכותה נסיעה מוסטת אחת. דירוג הספירות הללו נותן את ה**תחנות הסופגות**: התחנות שאכן יקבלו את התנועה אם תחנה זו תיסגר. זהו הפלט השימושי ביותר תפעולית במחברת כולה - הוא מציין היכן להוסיף קיבולת, ולא רק מה נשבר.

אם למועמדת אין תנועת מעבר במדגם, היא נרשמת עם `pairs_tested = 0` וסטטיסטיקות NaN במקום אפס מזויף. לאחר מכן היא מוצאת מהדירוגים ומהמתאם, וההוצאה מדווחת.

In [ ]:
# --- Shut down one station and re-route the trips that used it -------------
def reroute_one(v_idx):
    '''Remove node `v_idx` and re-route every sampled OD pair that used it.'''
    plist = through.get(int(v_idx), [])
    empty = {'pairs_tested': 0, 'pairs_reachable': 0, 'pairs_disconnected': 0,
             'detours': np.array([], dtype=float), 'absorb': {}, 'pair_ids': []}
    if not plist:
        return empty

    if len(plist) > MAX_PAIRS_PER_STATION:
        chosen = rng.choice(np.asarray(plist), size=MAX_PAIRS_PER_STATION, replace=False)
        sel = [int(p) for p in chosen]
    else:
        sel = list(plist)

    keep = (rows != v_idx) & (cols != v_idx)
    Gv = csr_matrix((data[keep], (rows[keep], cols[keep])), shape=(n_nodes, n_nodes))

    src_needed = sorted({pairs[p][0] for p in sel})
    row_of = {s: i for i, s in enumerate(src_needed)}
    dv, pv = dijkstra(Gv, directed=False, indices=src_needed, return_predecessors=True)

    detours, absorb, kept_ids = [], {}, []
    n_disc = 0
    for p in sel:
        s_i, t_i, base = pairs[p]
        r = row_of[s_i]
        new = dv[r, t_i]
        if not np.isfinite(new):
            n_disc += 1
            continue
        detours.append(max(0.0, float(new) - base))
        kept_ids.append(p)
        new_path = reconstruct(pv[r], s_i, t_i)
        if new_path is None:
            continue
        old = set(int(x) for x in old_paths[p])
        for nd in new_path[1:-1]:
            if nd not in old:
                absorb[nd] = absorb.get(nd, 0) + 1

    return {'pairs_tested': len(sel), 'pairs_reachable': len(detours),
            'pairs_disconnected': n_disc, 'detours': np.asarray(detours, dtype=float),
            'absorb': absorb, 'pair_ids': kept_ids}


records, absorb_rows, detour_rows = [], [], []
t_start = time.time()

for pos, row in enumerate(candidates.itertuples(index=False), start=1):
    sid = str(row.stop_id)
    v_idx = index_of[sid]
    out = reroute_one(v_idx)
    d = out['detours']
    tested = out['pairs_tested']

    top_absorb = sorted(out['absorb'].items(), key=lambda kv: kv[1], reverse=True)[:TOP_ABSORBING]
    label = ' | '.join(
        (name_of.get(node_ids[k], '') or node_ids[k]) + ' (' + str(node_ids[k]) + ') x' + str(c)
        for k, c in top_absorb)

    records.append({
        'removed_stop': sid,
        'stop_name': getattr(row, 'stop_name', '') or '',
        'mean_detour_seconds': float(d.mean()) if d.size else np.nan,
        'median_detour_seconds': float(np.median(d)) if d.size else np.nan,
        'pairs_disconnected': int(out['pairs_disconnected']),
        'reachable_share': (out['pairs_reachable'] / tested) if tested else np.nan,
        'top_absorbing_stops': label,
        'pairs_tested': tested,
        'pairs_reachable': int(out['pairs_reachable']),
        'p90_detour_seconds': float(np.percentile(d, 90)) if d.size else np.nan,
        'max_detour_seconds': float(d.max()) if d.size else np.nan,
        'zero_detour_share': float((d <= 1.0).mean()) if d.size else np.nan,
        'approx_betweenness': float(getattr(row, btw_col)),
        'topological_rank': int(row.topological_rank),
        'region': getattr(row, 'region', '') or '',
        'lat': getattr(row, 'lat', np.nan),
        'lon': getattr(row, 'lon', np.nan),
    })

    for k, c in sorted(out['absorb'].items(), key=lambda kv: kv[1], reverse=True):
        absorb_rows.append({'removed_stop': sid,
                            'removed_stop_name': getattr(row, 'stop_name', '') or '',
                            'absorbing_stop': node_ids[k],
                            'absorbing_stop_name': name_of.get(node_ids[k], ''),
                            'displaced_pairs': int(c)})

    for p, sec in zip(out['pair_ids'], d):
        detour_rows.append({'removed_stop': sid,
                            'origin_stop': node_ids[pairs[p][0]],
                            'destination_stop': node_ids[pairs[p][1]],
                            'baseline_seconds': round(pairs[p][2], 1),
                            'detour_seconds': round(float(sec), 1)})

    print(format(pos, '>3'), '/', len(candidates), sid,
          '| tested', format(tested, '>4'),
          '| disconnected', format(out['pairs_disconnected'], '>4'),
          '| mean detour', (format(d.mean() / 60, '.1f') + ' min') if d.size else 'n/a')

print()
print('rerouting sweep finished in', format(time.time() - t_start, '.1f'), 's')

## 10. עלות זמן-הנוסע המשוקללת, ושני הדירוגים

נדרש מספר יחיד לכל תחנה כדי לדרג אותן, וממוצע העיקוף לבדו אינו המספר הזה: תחנה שמנתקת מחצית מנסיעותיה עשויה להציג ממוצע עיקוף *נמוך*, פשוט משום שהזוגות שנפגעו קשה ביותר נשרו מהממוצע. לכן אנו מגדירים

```
expected_cost_seconds = reachable_share x mean_detour_seconds
                      + (1 - reachable_share) x DISCONNECT_PENALTY_S
```

כלומר תוספת זמן הנסיעה הצפויה של נסיעה משובשת אקראית, כאשר נסיעה שהופכת לבלתי אפשרית מחויבת ב-`DISCONNECT_PENALTY_S` קבוע (ברירת מחדל: שעה אחת). **קנס זה הוא הנחה ולא מדידה.** הוא נחשף כקבוע בדיוק כדי שניתן יהיה לערער עליו; בדיקת הרגישות שלהלן מדרגת מחדש את התחנות כאשר הקנס נקבע ל-30 דקות ול-3 שעות, ומדווחת אם סדר הדירוג שורד.

לאחר מכן אנו משווים את **דירוג זמן-הנוסע** עם **הדירוג הטופולוגי** (סדר ה-betweenness) באמצעות מקדם Spearman rho וחפיפת שתי קבוצות ה-top-10. אילו שני הדירוגים היו תואמים באופן מושלם, מחברת זו לא הייתה מוסיפה דבר על מחברת 04. מידת אי-ההסכמה ביניהם היא התרומה.

In [ ]:
# --- Composite cost, rankings and their agreement --------------------------
res = pd.DataFrame(records)

def expected_cost(frame, penalty):
    share = frame['reachable_share']
    mean_d = frame['mean_detour_seconds'].fillna(0.0)
    return share * mean_d + (1.0 - share) * penalty

res['expected_cost_seconds'] = expected_cost(res, DISCONNECT_PENALTY_S)

evaluated = res[res['pairs_tested'] > 0].copy()
skipped = len(res) - len(evaluated)
if evaluated.empty:
    raise RuntimeError('No candidate station carried any sampled through-traffic. '
                       'Raise N_SOURCES / TARGETS_PER_SOURCE and re-run section 8.')

evaluated = evaluated.sort_values('expected_cost_seconds', ascending=False).reset_index(drop=True)
evaluated['passenger_time_rank'] = np.arange(1, len(evaluated) + 1)
evaluated['rank_shift'] = evaluated['topological_rank'] - evaluated['passenger_time_rank']

rho, pval = spearmanr(evaluated['topological_rank'], evaluated['passenger_time_rank'])
k_top = min(10, len(evaluated))
top_topo = set(evaluated.nsmallest(k_top, 'topological_rank')['removed_stop'])
top_time = set(evaluated.nsmallest(k_top, 'passenger_time_rank')['removed_stop'])
overlap = len(top_topo & top_time)

# Sensitivity of the ordering to the disconnection penalty.
# Index [0] rather than .statistic, so this works on every scipy version.
sens = {}
for pen in (1800.0, DISCONNECT_PENALTY_S, 10800.0):
    alt = expected_cost(evaluated, pen).rank(ascending=False, method='min')
    sens['penalty_' + str(int(pen)) + 's_spearman_vs_default'] = round(
        float(spearmanr(alt, evaluated['passenger_time_rank'])[0]), 4)

print('stations evaluated                :', len(evaluated),
      '(' + str(skipped) + ' skipped: no sampled through-traffic)')
print('Spearman rho, topological vs time :', format(rho, '.3f'),
      '(p =', format(pval, '.4f') + ')')
print('top-' + str(k_top) + ' overlap                    :', overlap, 'of', k_top)
print('penalty sensitivity               :', sens)
print()
print('Biggest movers (topological rank -> passenger-time rank):')
movers = evaluated.reindex(evaluated['rank_shift'].abs().sort_values(ascending=False).index)
movers[['removed_stop', 'stop_name', 'topological_rank', 'passenger_time_rank',
        'rank_shift', 'mean_detour_seconds', 'reachable_share']].head(10).round(1)

## 11. כתיבת הטבלאות

נכתבות ארבע טבלאות, כולן בקידוד `utf-8-sig` כך שהשמות בעברית ייפתחו כראוי ב-Excel.

* **`rerouting_results.csv`** היא טבלת החוזה שמחברות אחרות צורכות. שבע עמודותיה הראשונות הן בדיוק `removed_stop, stop_name, mean_detour_seconds, median_detour_seconds, pairs_disconnected, reachable_share, top_absorbing_stops`; עמודות האבחון באות אחריהן.
* **`absorbing_stations.csv`** היא הצורה הארוכה של ספירות התחנות הסופגות - כל זוג (תחנה שהוסרה, תחנה סופגת), ולא רק חמש המובילות.
* **`rank_comparison.csv`** נושאת את שני הדירוגים זה לצד זה.
* **`detour_samples.csv`** היא הראיה הגולמית ברמת זוג OD בודד. זהו הקובץ שיש לפתוח אם צבר כלשהו שלעיל נראה בלתי סביר.

In [ ]:
# --- Persist the tables ----------------------------------------------------
CONTRACT = ['removed_stop', 'stop_name', 'mean_detour_seconds', 'median_detour_seconds',
            'pairs_disconnected', 'reachable_share', 'top_absorbing_stops']

results = res.merge(
    evaluated[['removed_stop', 'passenger_time_rank', 'rank_shift']],
    on='removed_stop', how='left')
ordered = CONTRACT + [c for c in results.columns if c not in CONTRACT]
results = results[ordered].sort_values('expected_cost_seconds', ascending=False)
results.to_csv(TABLES / 'rerouting_results.csv', index=False, encoding='utf-8-sig')

absorb_df = pd.DataFrame(absorb_rows, columns=['removed_stop', 'removed_stop_name',
                                               'absorbing_stop', 'absorbing_stop_name',
                                               'displaced_pairs'])
absorb_df = absorb_df.sort_values(['removed_stop', 'displaced_pairs'], ascending=[True, False])
absorb_df.to_csv(TABLES / 'absorbing_stations.csv', index=False, encoding='utf-8-sig')

rank_cmp = evaluated[['removed_stop', 'stop_name', 'approx_betweenness', 'topological_rank',
                      'expected_cost_seconds', 'passenger_time_rank', 'rank_shift',
                      'mean_detour_seconds', 'reachable_share', 'pairs_tested']]
rank_cmp.to_csv(TABLES / 'rank_comparison.csv', index=False, encoding='utf-8-sig')

detour_df = pd.DataFrame(detour_rows, columns=['removed_stop', 'origin_stop', 'destination_stop',
                                               'baseline_seconds', 'detour_seconds'])
detour_df.to_csv(TABLES / 'detour_samples.csv', index=False, encoding='utf-8-sig')

for path in ('rerouting_results.csv', 'absorbing_stations.csv',
             'rank_comparison.csv', 'detour_samples.csv'):
    print('wrote', TABLES / path)

results[CONTRACT].head(10)

## 12. איור 1 - עלות העיקוף לכל תחנה שהוסרה

תרשים הכותרת. עמודה אופקית אחת לכל תחנה מושבתת, כשאורכה = **ממוצע העיקוף בדקות** עבור הנסיעות שעדיין ניתן היה להשלים, ממוין לפי העלות הצפויה המשוקללת. צבע העמודה מקודד את **שיעור הנגישות** (`reachable_share`): אדום כהה משמעו שחלק גדול מהנסיעות המשובשות הפכו לבלתי אפשריות ולא רק לאיטיות יותר, וזהו כשל גרוע יותר מבחינה איכותית מאשר עיקוף ארוך. ההערה על כל עמודה מדווחת כמה מזוגות ה-OD שנבדקו נותקו.

יש לקרוא את שני הערוצים יחד. עמודה ארוכה ובהירה מייצגת בעיית *עומס* (כולם עדיין מגיעים, אך לאט). עמודה קצרה וכהה מייצגת בעיית *קשירות* (מעט עיקופים אפשריים בכלל). הן מחייבות התערבויות שונות.

In [ ]:
# --- Figure 1: detour cost per removed station -----------------------------
plot_df = evaluated.copy()
plot_df['mean_detour_min'] = plot_df['mean_detour_seconds'].fillna(0.0) / 60.0
plot_df['label'] = [((nm or '').strip() or sid) + '  (' + sid + ')'
                    for nm, sid in zip(plot_df['stop_name'], plot_df['removed_stop'])]
plot_df = plot_df.sort_values('expected_cost_seconds')      # largest ends on top

cmap = plt.get_cmap('RdYlGn')
colors = [cmap(float(s) if np.isfinite(s) else 1.0) for s in plot_df['reachable_share']]

fig, ax = plt.subplots(figsize=(11, max(6, 0.38 * len(plot_df))))
ax.barh(range(len(plot_df)), plot_df['mean_detour_min'], color=colors,
        edgecolor='#374151', linewidth=0.4)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['label'], fontsize=9)

span = max(plot_df['mean_detour_min'].max(), 1.0)
for y, (disc, tested) in enumerate(zip(plot_df['pairs_disconnected'], plot_df['pairs_tested'])):
    ax.text(plot_df['mean_detour_min'].iloc[y] + span * 0.012, y,
            str(int(disc)) + '/' + str(int(tested)) + ' cut',
            va='center', fontsize=8, color='#374151')

ax.set_xlim(0, span * 1.25)
ax.set_xlabel('Mean detour for trips that could still be completed (minutes)')
ax.set_title('Passenger-time cost of shutting one station\n'
             'top ' + str(len(plot_df)) + ' stations by betweenness, '
             'sorted by expected cost; colour = share of trips still reachable',
             fontsize=13, fontweight='bold')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])       # required by older matplotlib before it can draw a colorbar
plt.colorbar(sm, ax=ax, label='reachable share (green = everyone still gets there)')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES / 'detour_cost_per_station.png', dpi=FIG_DPI)
plt.show()
print('wrote', FIGURES / 'detour_cost_per_station.png')

## 13. איור 2 - הטופולוגיה אומרת דבר אחד, זמן הנוסע אומר אחר

כל נקודה היא תחנה אחת: ציר x = דירוגה לפי betweenness (1 = המרכזית ביותר), ציר y = דירוגה לפי עלות זמן-הנוסע הצפויה (1 = היקרה ביותר לאובדן). התאמה מושלמת הייתה ממקמת כל נקודה על האלכסון המקווקו. נקודות **מתחת** לאלכסון הן תחנות שהטופולוגיה *מעריכה בחסר* - הן מרכזיות פחות מאחרות אך עולות לנוסעים יותר כשהן נסגרות. נקודות **מעל** לאלכסון הן תחנות שהטופולוגיה *מעריכה ביתר*: מרכזיות מאוד, אך עם חלופות זולות ממש בסמוך, כך שהשבתתן עולה מעט זמן ממשי.

מקדם Spearman rho המודפס בכותרת מכמת את מידת ההתאמה הכוללת, והנקודות המתויגות הן אלה שזזו בצורה החדה ביותר. אם rho גבוה, המסקנה הכנה היא ש-betweenness כבר היה פרוקסי טוב, ושמחברת זו מוסיפה בעיקר את *סדר הגודל* (דקות) ולא סדר דירוג חדש - זהו ממצא לגיטימי, אם כי מרגש פחות, והוא מוצג ככזה במסקנות ולא מיופה.

In [ ]:
# --- Figure 2: topological rank vs passenger-time rank ---------------------
fig, ax = plt.subplots(figsize=(9, 8))
lim = len(evaluated) + 1
ax.plot([0, lim], [0, lim], '--', color='#9ca3af', linewidth=1.2,
        label='perfect agreement')

sc = ax.scatter(evaluated['topological_rank'], evaluated['passenger_time_rank'],
                s=90, c=evaluated['rank_shift'], cmap='coolwarm',
                edgecolor='#111827', linewidth=0.5, zorder=3)
plt.colorbar(sc, ax=ax, label='rank shift (topological - passenger-time)')

n_label = min(8, len(evaluated))
for _, r in movers.head(n_label).iterrows():
    ax.annotate((str(r['stop_name']).strip() or str(r['removed_stop'])),
                (r['topological_rank'], r['passenger_time_rank']),
                textcoords='offset points', xytext=(7, 5), fontsize=9, color='#111827')

ax.set_xlabel('Topological rank (1 = highest betweenness)')
ax.set_ylabel('Passenger-time rank (1 = most expensive closure)')
ax.set_title('Do the two definitions of critical agree?\n'
             'Spearman rho = ' + format(rho, '.3f') + ',  top-' + str(k_top) +
             ' overlap = ' + str(overlap) + '/' + str(k_top),
             fontsize=13, fontweight='bold')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.invert_xaxis()
ax.invert_yaxis()
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES / 'topological_vs_passenger_time_rank.png', dpi=FIG_DPI)
plt.show()
print('wrote', FIGURES / 'topological_vs_passenger_time_rank.png')

## 14. איור 3 - צורת התפלגות העיקופים

ממוצעים מסתירים את הזנב, והזנב הוא מה שאנשים מתלוננים עליו. היסטוגרמה זו מאגדת את כל זוגות ה-OD שנותבו מחדש מכל ההשבתות ומראה כיצד מתפלגת תוספת זמן הנסיעה. שני מאפיינים ראויים לבדיקה בפלט: הפסגה סמוך לאפס (נסיעות שמצאו חלופה מהירה באותה מידה ולא הושפעו כלל - הרשת ספגה את הסגירה) ואורך הזנב הימני (המיעוט של הנסיעות המשלמות קנס גדול מאוד). החציון והאחוזון ה-90 מסומנים בקווים אנכיים כך שהפער בין הנוסע הטיפוסי לנוסע חסר המזל גלוי לעין.

In [ ]:
# --- Figure 3: pooled detour distribution ----------------------------------
all_detours = detour_df['detour_seconds'].to_numpy(dtype=float) / 60.0
if all_detours.size == 0:
    print('No rerouted pairs were recorded - nothing to plot.')
else:
    cap = float(np.percentile(all_detours, 99))
    shown = all_detours[all_detours <= cap]
    med = float(np.median(all_detours))
    p90 = float(np.percentile(all_detours, 90))

    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.hist(shown, bins=60, color='#2563eb', edgecolor='white', linewidth=0.3)
    ax.axvline(med, color='#16a34a', linewidth=2,
               label='median = ' + format(med, '.1f') + ' min')
    ax.axvline(p90, color='#dc2626', linewidth=2, linestyle='--',
               label='90th percentile = ' + format(p90, '.1f') + ' min')
    ax.set_yscale('log')
    ax.set_xlabel('Extra travel time caused by the closure (minutes)')
    ax.set_ylabel('Number of OD pairs (log scale)')
    ax.set_title('How much extra travel time does one closed station impose?\n'
                 + format(len(all_detours), ',') + ' rerouted OD pairs across '
                 + str(len(evaluated)) + ' closures (x-axis capped at the 99th percentile)',
                 fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / 'detour_distribution.png', dpi=FIG_DPI)
    plt.show()

    print('detours (minutes): mean', format(all_detours.mean(), '.1f'),
          '| median', format(med, '.1f'),
          '| p90', format(p90, '.1f'),
          '| max', format(all_detours.max(), '.1f'))
    print('share of rerouted trips with a detour under 1 minute:',
          format(100 * float((all_detours < 1.0).mean()), '.1f') + '%')

## 15. איור 4 - מי סופג את העומס?

עבור ההשבתה היקרה ביותר, תרשים זה מציין בשמן את התחנות המופיעות במסלולים החדשים אך לא הופיעו בישנים, מדורגות לפי מספר זוגות ה-OD המוסטים שעוברים כעת דרכן. תפעולית, זהו הפלט השימושי ביותר במחברת: אם אותה תחנה מוצאת משירות, אלה התחנות שבהן יופיעו הנוסעים הנוספים והיכן יש למקם קיבולת, כוח אדם או שירות היסעים משלים.

הסתייגות כנה אחת לגבי משמעות הספירות: הן **ספירות מסלולים על פני זוגות ה-OD שנדגמו**, ולא נוסעים. הן מציינות *לאן* הולך העומס ומדרגות את המקומות הללו נכונה זה ביחס לזה; הן אינן מציינות כמה אנשים זה.

In [ ]:
# --- Figure 4: absorbing stations for the costliest closure ----------------
worst = evaluated.iloc[0]
worst_id = str(worst['removed_stop'])
worst_name = (str(worst['stop_name']).strip() or worst_id)
sub = absorb_df[absorb_df['removed_stop'] == worst_id].head(15).copy()

if sub.empty:
    print('No absorbing stations recorded for', worst_id, '- every rerouted trip kept its path.')
else:
    sub['label'] = [((nm or '').strip() or sid) + '  (' + str(sid) + ')'
                    for nm, sid in zip(sub['absorbing_stop_name'], sub['absorbing_stop'])]
    sub = sub.sort_values('displaced_pairs')

    fig, ax = plt.subplots(figsize=(10, max(5, 0.42 * len(sub))))
    ax.barh(range(len(sub)), sub['displaced_pairs'], color='#7c3aed')
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub['label'], fontsize=9)
    ax.set_xlabel('Displaced OD pairs now routed through this station')
    ax.set_title('Where the traffic goes when ' + worst_name + ' closes\n'
                 'stations newly appearing on the rerouted paths',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig(FIGURES / 'absorbing_stations_worst.png', dpi=FIG_DPI)
    plt.show()
    print('wrote', FIGURES / 'absorbing_stations_worst.png')

    print()
    print('Costliest closure:', worst_name, '(' + worst_id + ')')
    print('  mean detour     :', format(worst['mean_detour_seconds'] / 60, '.1f'), 'min')
    print('  reachable share :', format(worst['reachable_share'], '.3f'))
    print('  absorbed by     :', worst['top_absorbing_stops'])

## 16. קובץ הסיכום JSON

כל מה שמחברת מאוחרת יותר או הדוח הכתוב עשויים להזדקק לו, במילון אחד: הקבועים שהפיקו את ההרצה (כך שניתן תמיד לעקוב אחר מספר בחזרה אל ההגדרות שלו), גודל מדגם ה-OD, סטטיסטיקות העיקוף המאוגדות, נתוני ההתאמה בין הדירוגים, וזהות ההשבתה היקרה ביותר. אחסון הקבועים לצד התוצאות הוא מה שהופך את ההרצה לניתנת לשחזור ולא רק לניתנת לחזרה.

In [ ]:
# --- rerouting_summary.json ------------------------------------------------
pooled = detour_df['detour_seconds'].to_numpy(dtype=float)

summary = {
    'constants': {
        'TOP_N_STATIONS': TOP_N_STATIONS,
        'N_SOURCES': N_SOURCES,
        'TARGETS_PER_SOURCE': TARGETS_PER_SOURCE,
        'MAX_PAIRS_PER_STATION': MAX_PAIRS_PER_STATION,
        'MIN_EDGE_SECONDS': MIN_EDGE_SECONDS,
        'DISCONNECT_PENALTY_S': DISCONNECT_PENALTY_S,
        'RANDOM_SEED': RANDOM_SEED,
    },
    'graph': {
        'source_table': str(tt_path),
        'nodes': int(n_nodes),
        'undirected_edges': int(len(und)),
        'components': int(n_comp),
        'giant_component_nodes': int(len(giant_nodes)),
        'edges_clamped_to_min_seconds': int(n_clamped),
    },
    'od_sample': {
        'pairs_sampled': int(n_sampled),
        'pairs_through_a_candidate': int(len(pairs)),
        'share_through_a_candidate': round(len(pairs) / max(n_sampled, 1), 4),
    },
    'detours_seconds': {
        'rerouted_pairs': int(pooled.size),
        'mean': round(float(pooled.mean()), 1) if pooled.size else None,
        'median': round(float(np.median(pooled)), 1) if pooled.size else None,
        'p90': round(float(np.percentile(pooled, 90)), 1) if pooled.size else None,
        'max': round(float(pooled.max()), 1) if pooled.size else None,
        'share_under_60s': round(float((pooled < 60).mean()), 4) if pooled.size else None,
    },
    'disconnection': {
        'total_pairs_tested': int(res['pairs_tested'].sum()),
        'total_pairs_disconnected': int(res['pairs_disconnected'].sum()),
        'overall_reachable_share': round(
            1 - res['pairs_disconnected'].sum() / max(int(res['pairs_tested'].sum()), 1), 4),
    },
    'rank_agreement': {
        'stations_evaluated': int(len(evaluated)),
        'stations_skipped_no_traffic': int(skipped),
        'spearman_rho': round(float(rho), 4),
        'spearman_p_value': round(float(pval), 6),
        'top_k': int(k_top),
        'top_k_overlap': int(overlap),
        'max_abs_rank_shift': int(evaluated['rank_shift'].abs().max()),
        'penalty_sensitivity': sens,
    },
    'costliest_closure': {
        'stop_id': worst_id,
        'stop_name': str(worst['stop_name']),
        'expected_cost_seconds': round(float(worst['expected_cost_seconds']), 1),
        'mean_detour_seconds': (round(float(worst['mean_detour_seconds']), 1)
                                if np.isfinite(worst['mean_detour_seconds']) else None),
        'reachable_share': round(float(worst['reachable_share']), 4),
        'top_absorbing_stops': str(worst['top_absorbing_stops']),
    },
}

with open(STAGE / 'rerouting_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('wrote', STAGE / 'rerouting_summary.json')
print(json.dumps(summary, ensure_ascii=False, indent=2)[:1600])

## 17. מגבלות - לקרוא לפני ציטוט של כל מספר שלעיל

**1. אין הליכה, אין המתנה ואין קנס החלפה.** גרף זמני הנסיעה מכיל זמני נסיעה בתוך כלי הרכב בלבד. נסיעה מנותבת מחדש אמיתית עולה לנוסע הליכה אל התחנה החלופית, המתנה לשירות הבא, ולעתים קרובות החלפה נוספת - בדרך כלל מספר דקות כל אחת, ולעתים תכופות יותר מעיקוף הנסיעה המחושב כאן. כל עיקוף במחברת זו הוא אפוא **חסם תחתון** על עלות הנוסע האמיתית, והמסלול הקצר ביותר הוא *חסם עליון* על איכות החלופה (אף נוסע אינו יכול להשיגו, ורבים יעשו זאת גרוע יותר).

**2. תדירות אינה נלקחת בחשבון.** קו רכבת אחד ביום ואוטובוס כל ארבע דקות הם אותה קשת כאן. עיקוף המנותב אל שירות הפועל פעמיים ביום אינו, למעשה, עיקוף כלל. גרפי חלונות הזמן של מחברת 19 הם המקום הנכון לתקן זאת; שילוב זמן נסיעה עם תדירות שירות הוא ההרחבה הטבעית.

**3. זוגות OD אחידים על פני תחנות, לא על פני נוסעים.** לא מיושם מודל ביקוש, ולכן נסיעה בין שתי תחנות במדבר שוקלת כמו נסיעה בין שני מסופי החלפה בתל אביב. הדבר *מעריך בחסר* באופן שיטתי את עלות ההשבתות העירוניות, שבהן אותו עיקוף משפיע על הרבה יותר אנשים. הפרוקסי האוכלוסייתי ממחברת 21 הוא המכפיל החסר.

**4. קנס הניתוק הומצא.** `DISCONNECT_PENALTY_S = 3600` היא בחירה מודלית ללא בסיס אמפירי. הוא קיים משום שממוצע אינו יכול להכיל אינסוף. בדיקת הרגישות בפרק 10 מדווחת עד כמה זז הדירוג כאשר הוא מוקטן בחצי או מוכפל פי שלושה; יש לקרוא מספר זה לפני שסומכים על סדר הדירוג המשוקלל, ועדיף להעדיף את `mean_detour_seconds` יחד עם `reachable_share` כשני הפרימיטיבים הכנים.

**5. שגיאת דגימה.** עם `N_SOURCES = 250` ותקרה של `MAX_PAIRS_PER_STATION = 400`, הממוצעים לכל תחנה נשענים על כמה מאות זוגות לכל היותר; תחנות הנמצאות עמוק ברשימה עשויות להישען על מעט מאוד. `pairs_tested` מיוצא בדיוק מסיבה זו - אין לדרג בביטחון תחנה עם פחות מכ-50 זוגות שנבדקו אל מול שכנותיה. הניסוי כולו ניתן לשחזור באמצעות `RANDOM_SEED`, אך ניתן לשחזור אינו זהה למדויק.

**6. רק 30 התחנות הקריטיות טופולוגית נבדקות.** מכיוון שהמועמדות נשאבות מדירוג ה-betweenness, תחנה בעלת betweenness צנוע אך עלות זמן-נוסע קטסטרופלית אינה יכולה להופיע כאן - התכנון יכול להוריד בדירוג תחנה קריטית טופולוגית, אך לעולם לא לקדם תחנה שאינה מדורגת. אי-ההסכמה הנמדדת בפרק 10 היא אפוא אומדן *שמרני* של מידת הפער בין הטופולוגיה לזמן הנוסע.

**7. כשלים בודדים בלבד.** תחנה אחת בכל פעם. שיבושים אמיתיים (קו מוצף, שביתה, אירוע ביטחוני) מוציאים מכלל פעולה מסדרונות שלמים. כשל בו-זמני של תחנות מרובות הוא ניסוי אחר וקשה יותר.

**8. ספירות התחנות הסופגות הן ספירות מסלולים.** הן מזהות נכונה *לאן* מנותבות הנסיעות המוסטות ומדרגות את המקומות הללו, אך אינן נפחי נוסעים, והן יורשות במלואן את מגבלות 1-3.

## 18. מסקנות

* **פיצול הוא המקרה הנדיר; עיכוב הוא המקרה הרגיל.** על פני ההשבתות שנבדקו כאן, הרוב המכריע של זוגות ה-OD המשובשים נותרים נגישים - הם פשוט אורכים זמן רב יותר. מחברות החוסן הקודמות, המנקדות הסרה לפי מידת הצטמקות הרכיב הקשיר הגדול ביותר, מדווחות אפוא *דבר וחצי דבר* על רוב הנזק שסגירה גורמת. מדידת ההפסד בדקות ולא ברכיבים היא מה שהופך את ההשפעה לגלויה.

* **העלות מרוכזת בזנב.** חלק גדול מהנסיעות שנותבו מחדש מפסידות פחות מדקה - לרשת יש באמת חלופה מקבילה והיא סופגת את הסגירה - בעוד מיעוט קטן משלם מחיר כבד מאוד. זו הסיבה שחציון העיקוף וממוצע העיקוף שונים זה מזה, וזו הסיבה שהחציון לבדו היה גורם לכל סגירה להיראות חסרת נזק. יש לצטט את שניהם, או לצטט את האחוזון ה-90.

* **הטופולוגיה וזמן הנוסע מדרגים תחנות באופן שונה, וגודל הפער הזה הוא התוצאה.** יש לקרוא את מקדם Spearman rho ואת חפיפת ה-top-10 המודפסים בפרק 10 מתוך ההרצה שלכם עצמכם ולא מהזיכרון. תחנות שיורדות בדירוג זמן-הנוסע הן כאלה שיש בסמוך להן שירות מקביל טוב: מרכזיות בגרף, זולות לאובדן בפועל. תחנות שעולות הן כאלה שהחלופות שלהן הן עיקופים ארוכים או שאינן קיימות. אם rho יוצא גבוה, יש לומר זאת - המסקנה הכנה במקרה זה היא ש-betweenness הוא פרוקסי *סדר* סביר, ושתרומת המחברת היא *סדר הגודל*, בדקות, בתוספת התחנות הסופגות.

* **התחנות הסופגות הן הפלט השימושי ביותר תפעולית.** "אם תחנה זו נסגרת, חמש התחנות הללו מקבלות את התנועה" היא אמירה ברת-ביצוע באופן שציון betweenness אינו. `tables/absorbing_stations.csv` מכיל את המיפוי המלא, ולא רק את חמש המובילות לכל תחנה.

* **כל מספר כאן הוא רצפה, לא תחזית.** אין הליכה, אין המתנה, אין קנס החלפה, אין תדירות שירות, אין שקלול ביקוש, ויש קנס מומצא עבור ניתוק. המודל עונה על השאלה "בכמה ארוך יותר מסלול הרכב המהיר ביותר שנותר", שהיא הגרסה המצומצמת והכנה ביותר של השאלה. הכיוון של כל אפקט שהושמט זהה: העלות האמיתית לנוסעים גדולה יותר מזו המדווחת לעיל.